# 面试问题：MLA 为什么能压缩 KV Cache，工程上怎样判断能否使用低秩缓存？

可以直接复述的回答是：第一，普通 MHA 为每个 token 保存所有头的 K 和 V，缓存随头数与序列长度线性增长。第二，MLA 保存共享低维 latent，推理时再恢复各头 K/V。第三，低秩投影必须与训练权重共同学习，不能对任意模型无损压缩。第四，评估要同时看缓存字节、注意力输出误差和解码延迟。第五，异常请求可根据重构误差回退全量 KV。第六，RoPE 相关维度和量化元数据也要计入真实缓存。下面用五个客服会话和一个手写 PyTorch 缓存演示。

## 真实案例：客服 LLM 的长对话缓存容量规划

五条脱敏请求给出 prompt token、预计生成 token 和并发优先级。教学模型使用 4 个头、每头 4 维、latent rank 6 和 FP16 缓存；尺寸很小但字段与线上 Serving 一致。模型权重是固定随机种子生成的低秩权重，不代表真实基础模型精度。

In [1]:
requests = [  # 定义五条具有真实上下文长度的客服生成请求
    {"id": "CHAT-01", "topic": "退款进度", "prompt_tokens": 512, "decode_tokens": 96, "priority": "high"},  # 短上下文高优先级请求
    {"id": "CHAT-02", "topic": "合同条款总结", "prompt_tokens": 2048, "decode_tokens": 256, "priority": "normal"},  # 中长文档总结请求
    {"id": "CHAT-03", "topic": "多轮故障排查", "prompt_tokens": 4096, "decode_tokens": 384, "priority": "high"},  # 多轮诊断长上下文请求
    {"id": "CHAT-04", "topic": "月度工单归纳", "prompt_tokens": 8192, "decode_tokens": 512, "priority": "batch"},  # 批处理超长上下文请求
    {"id": "CHAT-05", "topic": "知识库问答", "prompt_tokens": 1024, "decode_tokens": 128, "priority": "normal"},  # 常规 RAG 问答请求
]  # 结束五条 Serving 输入
heads = 4  # 定义教学注意力头数
head_dim = 4  # 定义每个注意力头维度
latent_rank = 6  # 定义 MLA 共享 latent 维度
bytes_per_value = 2  # 使用 FP16 两字节估算缓存
print("请求输入：id | topic | prompt | decode | priority")  # 展示容量规划使用的真实字段
for request in requests:  # 逐条输出五个生成请求
    print(f"{request['id']} | {request['topic']:8} | {request['prompt_tokens']:5} | {request['decode_tokens']:3} | {request['priority']}")  # 呈现长度和优先级差异


请求输入：id | topic | prompt | decode | priority
CHAT-01 | 退款进度     |   512 |  96 | high
CHAT-02 | 合同条款总结   |  2048 | 256 | normal
CHAT-03 | 多轮故障排查   |  4096 | 384 | high
CHAT-04 | 月度工单归纳   |  8192 | 512 | batch
CHAT-05 | 知识库问答    |  1024 | 128 | normal


## Baseline / 基线：为每个头保存完整 K 和 V

普通 MHA 每个 token 保存 `2 × heads × head_dim` 个值。先按五条请求计算完整生命周期的缓存，得到后续方案的同口径基线。

In [2]:
def mha_cache_bytes(tokens):  # 计算普通多头注意力的单请求 KV 字节数
    return tokens * 2 * heads * head_dim * bytes_per_value  # 同时计入 K、V、所有头和 FP16 字节
baseline_bytes = {}  # 保存五条请求的完整 MHA 缓存需求
print("MHA 缓存：id | total_tokens | KiB")  # 输出逐请求容量基线
for request in requests:  # 按 prompt 加 decode 的最大生命周期估算
    total_tokens = request["prompt_tokens"] + request["decode_tokens"]  # 计算请求完成前最多驻留 token
    cache_bytes = mha_cache_bytes(total_tokens)  # 计算完整 KV 缓存字节数
    baseline_bytes[request["id"]] = cache_bytes  # 保存同口径结果供 MLA 对照
    print(f"{request['id']} | {total_tokens:5} | {cache_bytes / 1024:8.2f}")  # 展示缓存随上下文长度线性增长
print(f"五请求 MHA 合计：{sum(baseline_bytes.values()) / 1024:.2f} KiB")  # 汇总并发容量需求


MHA 缓存：id | total_tokens | KiB
CHAT-01 |   608 |    38.00
CHAT-02 |  2304 |   144.00
CHAT-03 |  4480 |   280.00
CHAT-04 |  8704 |   544.00
CHAT-05 |  1152 |    72.00
五请求 MHA 合计：1078.00 KiB


## 核心实现：共享 latent 缓存与按需恢复 K/V

为了单独验证缓存机制，构造本身具有 rank-6 结构的 K/V 权重。完整 MHA 与 latent 恢复使用同一组权重，因此注意力输出应一致。

In [3]:
import math  # 使用平方根缩放注意力分数
import torch  # 使用 PyTorch 张量实现底层投影和注意力
torch.manual_seed(2501)  # 固定权重和输入以保存确定性输出
hidden_dim = 16  # 定义教学隐藏维度
down = torch.randn(hidden_dim, latent_rank) / math.sqrt(hidden_dim)  # 创建隐藏状态到共享 latent 的低秩投影
up_k = torch.randn(latent_rank, heads * head_dim) / math.sqrt(latent_rank)  # 创建 latent 到各头 K 的恢复权重
up_v = torch.randn(latent_rank, heads * head_dim) / math.sqrt(latent_rank)  # 创建 latent 到各头 V 的恢复权重
w_q = torch.randn(hidden_dim, heads * head_dim) / math.sqrt(hidden_dim)  # 创建完整查询投影用于注意力计算
hidden = torch.randn(1, 12, hidden_dim)  # 模拟一个十二 token 的脱敏会话隐藏状态
latent_cache = hidden @ down  # 只把共享低维表示写入缓存
full_k = (latent_cache @ up_k).reshape(1, 12, heads, head_dim)  # 生成普通 MHA 会保存的完整 K
full_v = (latent_cache @ up_v).reshape(1, 12, heads, head_dim)  # 生成普通 MHA 会保存的完整 V
restored_k = (latent_cache @ up_k).reshape_as(full_k)  # 从 latent 在计算时恢复各头 K
restored_v = (latent_cache @ up_v).reshape_as(full_v)  # 从 latent 在计算时恢复各头 V
query = (hidden[:, -1] @ w_q).reshape(1, heads, head_dim)  # 投影最后一个 token 的四头查询
baseline_scores = torch.einsum("bhd,bshd->bhs", query, full_k) / math.sqrt(head_dim)  # 计算完整 KV 的注意力分数
latent_scores = torch.einsum("bhd,bshd->bhs", query, restored_k) / math.sqrt(head_dim)  # 计算恢复 KV 的注意力分数
baseline_output = torch.einsum("bhs,bshd->bhd", torch.softmax(baseline_scores, dim=-1), full_v)  # 计算 MHA 基线输出
latent_output = torch.einsum("bhs,bshd->bhd", torch.softmax(latent_scores, dim=-1), restored_v)  # 计算 MLA 恢复输出
max_output_error = float((baseline_output - latent_output).abs().max())  # 量化两种缓存路径的输出误差
print("缓存形状：full_k=", tuple(full_k.shape), "full_v=", tuple(full_v.shape), "latent=", tuple(latent_cache.shape))  # 展示每 token 缓存维度变化
print("最后 token 每头注意力 Top-1 位置：", latent_scores.argmax(dim=-1).tolist())  # 展示恢复后真正参与计算的中间结果
print(f"低秩构造下最大输出误差：{max_output_error:.3e}")  # 验证共享 latent 路径与完整 KV 一致


缓存形状：full_k= (1, 12, 4, 4) full_v= (1, 12, 4, 4) latent= (1, 12, 6)
最后 token 每头注意力 Top-1 位置： [[6, 2, 7, 5]]
低秩构造下最大输出误差：0.000e+00


## 失败案例与修正：任意 KV 强压到过低 rank 会失真

MLA 的低秩结构应随模型训练获得。若上线后直接对普通模型 KV 做 rank-1 SVD，注意力可能显著偏移。修正策略是离线校准 rank，并为重构误差异常的请求保留全量 KV 回退。

In [4]:
arbitrary_kv = torch.randn(12, heads * head_dim * 2)  # 构造不保证低秩结构的普通模型 K/V 矩阵
def svd_reconstruction(matrix, rank):  # 用截断 SVD 模拟上线后压缩任意 KV
    left, singular, right = torch.linalg.svd(matrix, full_matrices=False)  # 分解 token 与 KV 维度的主要方向
    restored = (left[:, :rank] * singular[:rank]) @ right[:rank]  # 仅使用指定 rank 重构原矩阵
    relative_error = float(torch.linalg.norm(matrix - restored) / torch.linalg.norm(matrix))  # 计算相对重构误差
    return restored, relative_error  # 返回重构值和可用于门禁的误差
rank_rows = []  # 收集不同 rank 的误差和缓存决策
for rank in (1, 3, 6, 10):  # 比较过低到较高的四个 latent 维度
    restored, error = svd_reconstruction(arbitrary_kv, rank)  # 对同一任意 KV 执行截断重构
    storage = 12 * rank + rank * arbitrary_kv.shape[1]  # 估算 latent 加恢复基的元素数
    decision = "fallback_full_kv" if error > 0.45 else "use_latent"  # 超过教学误差阈值时回退完整缓存
    rank_rows.append((rank, error, storage, decision))  # 保存误差、容量和决策
print("rank | relative_error | stored_values | decision")  # 输出压缩率与精度的真实权衡
for rank, error, storage, decision in rank_rows:  # 逐项展示四个 rank 的结果
    print(f"{rank:2} | {error:.3f} | {storage:4} | {decision}")  # 显示过低 rank 的失败和回退动作


rank | relative_error | stored_values | decision
 1 | 0.901 |   44 | fallback_full_kv
 3 | 0.729 |  132 | fallback_full_kv
 6 | 0.515 |  264 | fallback_full_kv
10 | 0.239 |  440 | use_latent


## 结果表：五条请求的 MHA 与 MLA 缓存对照

In [5]:
def mla_cache_bytes(tokens):  # 计算仅保存共享 latent 的教学缓存字节数
    return tokens * latent_rank * bytes_per_value  # 不把模型常驻恢复权重重复计入每请求缓存
mla_bytes = {}  # 保存五条请求的 latent 缓存需求
print("id | MHA_KiB | MLA_KiB | reduction | saved_KiB")  # 输出同请求同 token 数的容量对照
for request in requests:  # 对五条 Serving 请求逐一比较
    total_tokens = request["prompt_tokens"] + request["decode_tokens"]  # 使用与基线相同的最大生命周期 token
    compressed = mla_cache_bytes(total_tokens)  # 计算共享 latent 字节数
    original = baseline_bytes[request["id"]]  # 读取完整 KV 基线字节数
    mla_bytes[request["id"]] = compressed  # 保存压缩结果供汇总测试
    reduction = 1 - compressed / original  # 计算相对缓存节省比例
    print(f"{request['id']} | {original / 1024:8.2f} | {compressed / 1024:8.2f} | {reduction:8.1%} | {(original - compressed) / 1024:8.2f}")  # 展示每条请求的容量收益
total_reduction = 1 - sum(mla_bytes.values()) / sum(baseline_bytes.values())  # 汇总五请求并发缓存节省比例
print(f"并发合计缓存下降：{total_reduction:.1%}")  # 输出容量规划最关心的汇总指标


id | MHA_KiB | MLA_KiB | reduction | saved_KiB
CHAT-01 |    38.00 |     7.12 |    81.2% |    30.88
CHAT-02 |   144.00 |    27.00 |    81.2% |   117.00
CHAT-03 |   280.00 |    52.50 |    81.2% |   227.50
CHAT-04 |   544.00 |   102.00 |    81.2% |   442.00
CHAT-05 |    72.00 |    13.50 |    81.2% |    58.50
并发合计缓存下降：81.2%


## 结果解读

在 4 头、head_dim 4 的教学配置中，MHA 每 token 保存 32 个值，latent 只保存 6 个值，因此五条请求的缓存下降 81.2%。对训练得到的低秩权重，恢复路径误差为零；对任意 KV 的 rank-1 后压缩，重构误差触发回退。容量收益不能脱离模型是否按 MLA 结构训练这一前提。

## 生产边界

真实 MLA 还涉及 query 压缩、解耦 RoPE、权重吸收、分页块对齐、量化 scale 和 GPU kernel 融合。缓存字节需计入 block metadata、碎片和并发调度；误差门禁应在真实任务精度上校准，而非只看矩阵范数。本例没有测 GPU 带宽、首 token 延迟和长上下文精度。

## 最小回归测试

In [6]:
assert len(requests) >= 5  # 保证容量案例覆盖至少五条真实生成请求
assert latent_cache.shape[-1] == latent_rank  # 保证请求缓存实际写入共享低维 latent
assert max_output_error < 1e-7  # 保证低秩构造模型的恢复注意力与完整 KV 一致
assert rank_rows[0][3] == "fallback_full_kv"  # 保证任意 KV 的过低 rank 触发精度回退
assert all(mla_bytes[key] < baseline_bytes[key] for key in mla_bytes)  # 保证五条请求的 latent 缓存均小于 MHA
assert total_reduction > 0.8  # 保证教学配置的并发缓存下降超过八成
